# Squeeze regimes — GEX+ scarcity / stress–calm

**Paper:** GEX+ · abundance/scarce · squeeze/stress (PDF pp. 9–12) · **NOTES:** [`NOTES.md`](NOTES.md) · **EXP:** [`EXP_REPORT.md`](EXP_REPORT.md)  
**Artifacts:** `out/squeeze_regimes/`

Regime split: stress = above-median `|squeeze_intensity|` vs calm. Scarcity from GEX+ sign/threshold. **Hold** — not a Promote path.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / "research" / "books" / "squeeze_metrics"
OUT = BOOK / "out"
SCRIPTS = BOOK / "scripts"
ROOT = BOOK.parents[2]
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(ROOT))
from _nb_common import (
    load_json, gex_day_table, safe_corr, partial_corr_vs_rv, lead_lag_day, SIGNAL_BOARD,
)
from certified_panel import load_certified, primary_days, spot_l2_days, gex_panel_days, DDOI_LABEL, GEX_PROXY_LABEL
from research.lib import squeeze  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

def show_pngs(fig_dir: Path, pattern="*.png"):
    if not fig_dir.is_dir():
        print("no figs dir", fig_dir)
        return
    for p in sorted(fig_dir.glob(pattern)):
        display(Markdown(f"**{p.name}** — `{p.relative_to(BOOK)}`"))
        display(Image(filename=str(p)))

def print_board(rows=None):
    rows = rows or SIGNAL_BOARD
    print(f"{'id':34s} {'gate':6s} note")
    print("-" * 88)
    for i, g, n in rows:
        print(f"{i:34s} {g:6s} {n}")

CERT = load_certified()
PRIMARY = primary_days(CERT)
SPOT = spot_l2_days(CERT)
GEX_DAYS = gex_panel_days(CERT)
print("BOOK", BOOK)
print("certified primary n=", len(PRIMARY), PRIMARY)
print("panel_gex_options n=", len(GEX_DAYS), GEX_DAYS)
print("spot_l2 subpanel n=", len(SPOT), SPOT)
print("DDOI label", DDOI_LABEL, "| GEX proxy", GEX_PROXY_LABEL)
print("trade_synth QUARANTINED — never primary; warehouse OI futures-only unused")


In [ ]:

summary = load_json(OUT / "squeeze_regimes" / "summary.json")
panel = load_json(OUT / "squeeze_regimes" / "panel_rows.json") or (summary or {}).get("rows") or []
tbl = gex_day_table(panel, CERT)
print(json.dumps({k: (summary or {}).get(k) for k in (
    "n_ok","days_ok","n_scarce","gex_plus_mean","regime_split_gexplus_vs_range","decision","promote",
)}, indent=2, default=str))
display(tbl.round(5) if len(tbl) else tbl)


## Scarcity map + stress/calm corr


In [ ]:

ok = tbl[tbl["ok"] == True].copy() if len(tbl) else tbl
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.0))
if len(ok):
    colors = ["#ee6c4d" if bool(s) else "#98c1d9" for s in ok["scarce"]]
    axes[0].bar(ok["day"], ok["gex_plus"], color=colors)
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].tick_params(axis="x", rotation=70)
    axes[0].set_title("GEX+ (red=scarce)")
    axes[1].scatter(ok["squeeze_intensity"], ok["hl_range"], s=55, c=ok["gex_plus"], cmap="coolwarm")
    axes[1].set_xlabel("squeeze intensity"); axes[1].set_ylabel("HL range")
    axes[1].set_title("Intensity vs range")
    med = float(ok["squeeze_intensity"].median()) if len(ok) else np.nan
    ok["regime"] = np.where(ok["squeeze_intensity"] >= med, "stress", "calm")
    for lab, c in (("calm", "#98c1d9"), ("stress", "#ee6c4d")):
        g = ok[ok["regime"] == lab]
        axes[2].scatter(g["gex_plus"], g["hl_range"], s=55, c=c, label=f"{lab} n={len(g)}")
    axes[2].set_xlabel("GEX+"); axes[2].set_ylabel("HL range"); axes[2].legend(fontsize=8)
    axes[2].set_title("Stress/calm (intensity median split)")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "run exp_squeeze_regimes.py", ha="center", transform=ax.transAxes)
plt.tight_layout(); plt.show()
print("regime_split artifact:", (summary or {}).get("regime_split_gexplus_vs_range"))
if len(ok):
    print(ok.groupby("regime")[["gex_plus","hl_range","hl_rv"]].mean().round(6))


## Signal board


In [ ]:

print_board([
    ("risk.squeeze_intensity", "Hold", f"n_ok={(summary or {}).get('n_ok')} n_scarce={(summary or {}).get('n_scarce')}"),
    ("liq.implied_book_scarcity", "Hold", "scarce flags from GEX+"),
    ("alpha.tob_cross_arb", "Kill", "never Promote"),
])
